# Week 3 — Wine Classifier Showdown

**Theme:** Supervised learning II — decision trees, ensembles, neural networks

Last week we used k-NN and linear regression. This week we try three more
supervised learners on the same task and compare them head-to-head:

- **Decision Tree** — learns a sequence of yes/no questions ("is alcohol > 12.9?")
- **Random Forest** — an *ensemble* of many different decision trees that vote together
- **Neural Network (MLP)** — a small multi-layer perceptron

**Dataset:** scikit-learn's built-in `wine` dataset — 178 wines, 13 chemical
measurements each, 3 grape-growing regions (cultivars) to classify.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target
print("Features:", list(wine.feature_names))
print("Classes:", list(wine.target_names))
print("Shape:", X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Neural networks are sensitive to feature scale, so we standardize
# (mean 0, std 1). Trees and forests don't need this, but it doesn't hurt them.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Model 1: Decision Tree

A tree of simple yes/no questions, learned automatically from the data.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
tree_acc = accuracy_score(y_test, tree.predict(X_test))
print(f"Decision tree test accuracy: {tree_acc:.2%}")

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(tree, feature_names=wine.feature_names, class_names=wine.target_names,
          filled=True, fontsize=9)
plt.title("Decision Tree (max_depth=3)")
plt.show()

## Model 2: Random Forest (ensemble)

A single tree can overfit or be unstable. A **random forest** trains many trees,
each on a slightly different random subset of the data and features, then has
them vote. This usually generalizes better than any single tree.

In [ ]:
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
forest_acc = accuracy_score(y_test, forest.predict(X_test))
print(f"Random forest test accuracy: {forest_acc:.2%}")

In [ ]:
# Which chemical measurements did the forest find most useful?
import pandas as pd
importances = pd.Series(forest.feature_importances_, index=wine.feature_names)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(6, 5))
importances.plot(kind="barh", color="seagreen")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.show()

## Model 3: Neural Network (MLPClassifier)

A small feedforward neural network: an input layer (13 features), one hidden
layer of artificial neurons, and an output layer (3 classes). This is the same
basic idea we'll build from scratch in PyTorch in Week 9 — here we use
scikit-learn's ready-made version.

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000, random_state=42)
mlp.fit(X_train_scaled, y_train)
mlp_acc = accuracy_score(y_test, mlp.predict(X_test_scaled))
print(f"Neural network test accuracy: {mlp_acc:.2%}")

## Compare all three

In [ ]:
results = pd.Series(
    {"Decision Tree": tree_acc, "Random Forest": forest_acc, "Neural Network": mlp_acc}
)
print(results)

plt.figure(figsize=(6, 4))
results.plot(kind="bar", color=["indianred", "seagreen", "steelblue"])
plt.title("Test Accuracy by Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.show()

## Try it yourself

1. **Overfit a tree on purpose.** Set `max_depth=None` (unlimited) on a new
   `DecisionTreeClassifier` — does test accuracy go up or down compared to
   `max_depth=3`? Why might a deeper tree memorize the training data instead
   of learning general patterns?
2. **More trees, better forest?** Try `n_estimators=5` vs. `n_estimators=200`
   in the random forest — does accuracy keep improving, or does it plateau?
3. **Bigger network.** Try `hidden_layer_sizes=(32, 16)` (two hidden layers)
   in the MLP. Does it help on a dataset this small?
4. **Which model would you actually deploy?** Beyond accuracy, which model is
   easiest to explain to a non-technical wine buyer? Which trains fastest?

---
## 🎯 캡스톤: 오늘 수업 갈까 말까? 출석 예측기

200일치 가상의 "그날 상황 -> 실제 등교 여부" 더미 기록을 드립니다. Part 1-3에서 배운 **결정 트리 / 랜덤 포레스트 / 신경망**을 직접 학습시켜서, 어떤 요인이 결석에 가장 큰 영향을 주는지 알아내고, "오늘" 상황을 입력하면 등교할지 예측해보는 모델을 만들어보세요.

**확장 아이디어:** 한 학기 동안 실제로 본인의 등교 여부와 그날 상황(수면시간, 날씨, 과제 유무 등)을 기록해서 `attendance_df`를 바꿔치기하면 진짜 "내 출석 습관 분석기"가 됩니다.

In [ ]:
# 더미 데이터 생성 (실행만 하면 됩니다)
import pandas as pd
rng = np.random.default_rng(3)
n_days = 200

hours_slept = np.clip(rng.normal(6.5, 1.5, n_days), 2, 11)
weather_bad = rng.binomial(1, 0.25, n_days)
assignment_due_today = rng.binomial(1, 0.2, n_days)
first_class_hour = rng.integers(9, 19, n_days)
motivation_score = np.clip(rng.normal(6, 2, n_days), 1, 10)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

logit = (
    -3.0
    + 0.5 * hours_slept
    - 1.8 * weather_bad
    - 0.5 * assignment_due_today
    + 0.35 * motivation_score
    - 0.8 * (first_class_hour < 10).astype(float)
)
attend_prob = sigmoid(logit)
attended = (rng.random(n_days) < attend_prob).astype(int)

attendance_df = pd.DataFrame({
    "hours_slept": hours_slept.round(1),
    "weather_bad": weather_bad,
    "assignment_due_today": assignment_due_today,
    "first_class_hour": first_class_hour,
    "motivation_score": motivation_score.round(1),
    "attended": attended,
})
print("결석 비율:", (1 - attendance_df["attended"]).mean().round(2))
attendance_df.head()

### 여러분의 과제

1. `hours_slept`, `weather_bad`, `assignment_due_today`, `first_class_hour`, `motivation_score`를 입력(X), `attended`를 정답(y)으로 하여 **DecisionTreeClassifier**, **RandomForestClassifier**, **MLPClassifier**(신경망은 `StandardScaler`로 스케일링 필요) 세 모델을 각각 학습시키고 테스트 정확도를 비교하세요. (위 Model 1~3 코드를 참고하세요.)
2. 랜덤 포레스트의 `feature_importances_`를 큰 순서대로 출력해서, **어떤 요인이 결석에 가장 큰 영향을 주는지** 확인하세요.
3. 아래 `my_day`에 오늘의 상황을 직접 입력하고, 학습된 모델 중 하나로 "오늘 등교할 것 같은지" 예측해보세요.

In [ ]:
# TODO 1: 세 모델(Decision Tree, Random Forest, MLP)을 학습시키고 테스트 정확도를 비교하세요.


# TODO 2: 랜덤 포레스트의 feature_importances_를 큰 순서대로 출력해보세요.


# TODO 3: 오늘의 상황을 입력하고 등교 여부를 예측해보세요.
my_day = {
    "hours_slept": None,          # 예: 5.5
    "weather_bad": None,           # 예: 0 또는 1
    "assignment_due_today": None,  # 예: 0 또는 1
    "first_class_hour": None,      # 예: 9
    "motivation_score": None,      # 예: 4
}